# SVM para clasificación de resultados electorales 2020 (EE.UU. por condado)

**Proyecto:** SVM EC2 UCSUR  
**Plan:** [`IMPLEMENTATION-PLAN.md`](IMPLEMENTATION-PLAN.md)  
**Requerimientos:** [`REQUERIMENTS.md`](REQUERIMENTS.md)

---

## Fase 1 — Carga y exploración de datos

**Objetivo:** confirmar que el dataset está sano y dimensionado para SVM (~3,100 condados).

Decisiones técnicas aplicadas (ver `IMPLEMENTATION-PLAN.md` §0):
- Features candidatas: `votes_gop`, `votes_dem`, `total_votes` (las únicas sin riesgo de data leakage).
- Clase objetivo `winner`: `GOP` si `per_point_diff > 0`, `DEM` en caso contrario.
- Fuente: [`tonmcg/US_County_Level_Election_Results_08-24`](https://github.com/tonmcg/US_County_Level_Election_Results_08-24).

### 1.1 Imports y configuración

In [1]:
import os
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', 50)

RANDOM_STATE = 42
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
CSV_PATH = DATA_DIR / '2020_US_County_Level_Presidential_Results.csv'
CSV_URL = ('https://raw.githubusercontent.com/tonmcg/'
           'US_County_Level_Election_Results_08-24/master/'
           '2020_US_County_Level_Presidential_Results.csv')

print('Setup OK. Versiones:')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')

Setup OK. Versiones:
  pandas  : 3.0.3
  numpy   : 2.4.6


### 1.2 Descarga (con cache) y lectura del CSV

Si el archivo ya existe localmente en `data/`, se reutiliza (cache). Si no, se descarga desde el raw de GitHub.

In [2]:
if not CSV_PATH.exists():
    print(f'Descargando {CSV_URL} ...')
    urlretrieve(CSV_URL, CSV_PATH)
    print(f'Guardado en {CSV_PATH} ({CSV_PATH.stat().st_size / 1024:.1f} KB)')
else:
    print(f'Usando cache local: {CSV_PATH} ({CSV_PATH.stat().st_size / 1024:.1f} KB)')

df = pd.read_csv(CSV_PATH)
print(f'\nDimensiones: {df.shape[0]:,} filas x {df.shape[1]} columnas')

Usando cache local: data/2020_US_County_Level_Presidential_Results.csv (339.7 KB)

Dimensiones: 3,152 filas x 10 columnas


### 1.3 Inspección inicial: tipos, nulos, primeras filas

In [3]:
print('=== TIPOS DE DATOS ===')
print(df.dtypes)
print()
print('=== VALORES NULOS POR COLUMNA ===')
print(df.isnull().sum())
print()
print('=== PRIMERAS 5 FILAS ===')
df.head()

=== TIPOS DE DATOS ===
state_name            str
county_fips         int64
county_name           str
votes_gop           int64
votes_dem           int64
total_votes         int64
diff                int64
per_gop           float64
per_dem           float64
per_point_diff    float64
dtype: object

=== VALORES NULOS POR COLUMNA ===
state_name        0
county_fips       0
county_name       0
votes_gop         0
votes_dem         0
total_votes       0
diff              0
per_gop           0
per_dem           0
per_point_diff    0
dtype: int64

=== PRIMERAS 5 FILAS ===


,state_name,county_fips,county_name,votes_gop,votes_dem,total_votes,diff,per_gop,per_dem,per_point_diff
0,Alabama,1001,Autauga County,19838,7503,27770,12335,0.71,0.27,0.44
1,Alabama,1003,Baldwin County,83544,24578,109679,58966,0.76,0.22,0.54
2,Alabama,1005,Barbour County,5622,4816,10518,806,0.53,0.46,0.08
3,Alabama,1007,Bibb County,7525,1986,9595,5539,0.78,0.21,0.58
4,Alabama,1009,Blount County,24711,2640,27588,22071,0.90,0.10,0.80


### 1.4 Distribución de la clase provisional `winner`

Construimos `winner` provisionalmente (sin guardar la columna todavía, solo para inspección) a partir de `per_point_diff` para confirmar la proporción esperada de condados GOP vs DEM.

In [4]:
winner_provisional = np.where(df['per_point_diff'] > 0, 'GOP', 'DEM')
dist = pd.Series(winner_provisional).value_counts(normalize=True).mul(100).round(2)
print('=== DISTRIBUCIÓN DE CLASES (provisional) ===')
print(dist.astype(str) + ' %')
print()
print('Conteos absolutos:')
print(pd.Series(winner_provisional).value_counts())

=== DISTRIBUCIÓN DE CLASES (provisional) ===
GOP    82.33 %
DEM    17.67 %
Name: proportion, dtype: str

Conteos absolutos:
GOP    2595
DEM     557
Name: count, dtype: int64


**Interpretación esperada:** aproximadamente 75% GOP y 25% DEM a nivel condado (sesgo rural). Esto confirma el desbalance y justifica el uso de `stratify=y` en la partición de la Fase 2, así como reportar métricas por clase (no solo accuracy global).

### 1.5 Estadísticas descriptivas de las 3 features candidatas

In [5]:
feature_cols = ['votes_gop', 'votes_dem', 'total_votes']
df[feature_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
votes_gop,"3,152.00","23,543.21","54,039.94",60.00,"3,661.50","8,123.00","20,508.75","1,145,530.00"
votes_dem,"3,152.00","25,782.04","96,930.23",4.00,"1,319.50","3,690.00","11,944.50","3,028,885.00"
total_votes,"3,152.00","50,264.45","149,379.19",66.00,"5,414.75","12,335.50","33,304.50","4,263,443.00"


### 1.6 Resumen de la Fase 1

**Lo que se valida aquí:**
- `df.shape` debe ser cercano a `(~3112, 10)`.
- Las 3 features (`votes_gop`, `votes_dem`, `total_votes`) deben ser numéricas y sin nulos.
- La proporción GOP/DEM debe ser coherente con la realidad electoral 2020 a nivel condado (~75/25).

Si los números anteriores son consistentes, se puede avanzar a la **Fase 2 (Preprocesamiento)**.

---

## Fase 2 — Preprocesamiento

**Objetivo:** producir `X_train`, `X_test`, `y_train`, `y_test` listos para SVM.

Pasos:
1. Crear la clase nominal `winner` desde `per_point_diff`.
2. Verificar nulos en features y clase.
3. Definir `X` (3 features crudas) e `y` (clase).
4. Split 80/20 **estratificado** con `random_state=42`.
5. `StandardScaler` (fit en train, transform en test).

### 2.1 Crear la clase `winner`

In [6]:
df['winner'] = np.where(df['per_point_diff'] > 0, 'GOP', 'DEM')
print('Conteo de la clase winner:')
print(df['winner'].value_counts())
print()
print('Proporción:')
print(df['winner'].value_counts(normalize=True).mul(100).round(2).astype(str) + ' %')

Conteo de la clase winner:
winner
GOP    2595
DEM     557
Name: count, dtype: int64

Proporción:
winner
GOP    82.33 %
DEM    17.67 %
Name: proportion, dtype: str


### 2.2 Verificar nulos en features y en la clase

In [7]:
nulos_features = df[feature_cols].isnull().sum().sum()
nulos_clase     = df['winner'].isnull().sum()
print(f'Nulos en features ({feature_cols}): {nulos_features}')
print(f'Nulos en winner: {nulos_clase}')
assert nulos_features == 0 and nulos_clase == 0, 'Hay nulos a tratar'
print('OK: sin nulos en features ni en winner.')

Nulos en features (['votes_gop', 'votes_dem', 'total_votes']): 0
Nulos en winner: 0
OK: sin nulos en features ni en winner.


### 2.3 Definir X e y, y split 80/20 estratificado

In [8]:
from sklearn.model_selection import train_test_split

X = df[feature_cols].values
y = df['winner'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')
print(f'y_train: {y_train.shape}  y_test: {y_test.shape}')
print()
print('Proporción de clases en y_train:', np.round(np.mean(y_train == 'GOP') * 100, 2), '% GOP')
print('Proporción de clases en y_test :', np.round(np.mean(y_test  == 'GOP') * 100, 2), '% GOP')
print()
print('La estratificación garantiza que la proporción GOP/DEM sea ~idéntica en train y test.')

X_train: (2521, 3)  X_test: (631, 3)
y_train: (2521,)  y_test: (631,)

Proporción de clases en y_train: 82.35 % GOP
Proporción de clases en y_test : 82.25 % GOP

La estratificación garantiza que la proporción GOP/DEM sea ~idéntica en train y test.


### 2.4 Escalado con `StandardScaler` (fit en train, transform en test)

In [9]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

print('Medias de X_train (post-escalado):', np.round(X_train.mean(axis=0), 6).tolist())
print('Stds  de X_train (post-escalado):', np.round(X_train.std(axis=0),  6).tolist())
print()
print('X_train[:3] (muestra):')
print(X_train[:3])

Medias de X_train (post-escalado): [-0.0, -0.0, 0.0]
Stds  de X_train (post-escalado): [1.0, 1.0, 1.0]

X_train[:3] (muestra):
[[-0.42830318 -0.25584744 -0.32388481]
 [ 0.03375637 -0.03974611 -0.01520572]
 [-0.4310643  -0.25738035 -0.32584962]]


### 2.5 Resumen de la Fase 2

- ✅ Clase `winner` creada: 82.33% GOP, 17.67% DEM.
- ✅ Sin nulos en features ni en la clase.
- ✅ Split 80/20 estratificado (`random_state=42`): ~2521 train / ~631 test.
- ✅ Features estandarizadas (media 0, std 1 en train; misma transformación aplicada a test).

**Listo para entrenar el SVM en la Fase 3.**

---

## Fase 3 — Entrenamiento del SVM lineal

**Objetivo:** entrenar 5 modelos con diferentes valores de `C` para mostrar la transición margen blando → duro.

- Algoritmo: `sklearn.svm.SVC(kernel='linear')`.
- `C` controla la penalización por errores en entrenamiento:
  - **C bajo (margen blando):** permite más errores, margen más ancho, mejor generalización esperada.
  - **C alto (margen duro):** castiga mucho los errores, margen más estrecho, riesgo de overfitting.

### 3.1 Entrenar los 5 modelos

In [10]:
from sklearn.svm import SVC

C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
models = {f'C={c}': SVC(kernel='linear', C=c, random_state=RANDOM_STATE) for c in C_values}

for name, model in models.items():
    model.fit(X_train, y_train)
    print(f'Entrenado {name:>8}  |  n_support = {model.n_support_.tolist()}  |  total SV = {model.support_.shape[0]}')

Entrenado   C=0.01  |  n_support = [390, 390]  |  total SV = 780
Entrenado    C=0.1  |  n_support = [325, 326]  |  total SV = 651


Entrenado    C=1.0  |  n_support = [258, 258]  |  total SV = 516
Entrenado   C=10.0  |  n_support = [183, 183]  |  total SV = 366
Entrenado  C=100.0  |  n_support = [114, 113]  |  total SV = 227


---

## Fase 4 — Evaluación

Métricas por modelo:
- `accuracy` global.
- `precision`, `recall`, `f1` **por clase** (GOP y DEM) — importante por el desbalance.
- `confusion_matrix`.
- Número de vectores de soporte.

### 4.1 Métricas por modelo (tabla comparativa)

In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

rows = []
for name, model in models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    rows.append({
        'Modelo'      : name,
        'Accuracy'    : round(acc, 4),
        'Prec_GOP'    : round(report['GOP']['precision'], 4),
        'Recall_GOP'  : round(report['GOP']['recall'], 4),
        'F1_GOP'      : round(report['GOP']['f1-score'], 4),
        'Prec_DEM'    : round(report['DEM']['precision'], 4),
        'Recall_DEM'  : round(report['DEM']['recall'], 4),
        'F1_DEM'      : round(report['DEM']['f1-score'], 4),
        'N_SV'        : int(model.support_.shape[0]),
    })

df_results = pd.DataFrame(rows)
df_results

,Modelo,Accuracy,Prec_GOP,Recall_GOP,F1_GOP,Prec_DEM,Recall_DEM,F1_DEM,N_SV
0,C=0.01,0.86,0.86,1.00,0.92,0.93,0.23,0.37,780
1,C=0.1,0.88,0.88,1.00,0.93,0.98,0.36,0.52,651
2,C=1.0,0.90,0.89,1.00,0.94,0.98,0.46,0.62,516
3,C=10.0,0.94,0.93,1.00,0.96,1.00,0.66,0.80,366
4,C=100.0,0.98,0.98,1.00,0.99,1.00,0.90,0.95,227


### 4.2 Matriz de confusión del mejor modelo (mayor F1-DEM)

In [12]:
best_row = df_results.loc[df_results['F1_DEM'].idxmax()]
best_name = best_row['Modelo']
best_model = models[best_name]

y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best, labels=['GOP', 'DEM'])

print(f'Mejor modelo según F1-DEM: {best_name}')
print(f'Accuracy = {best_row["Accuracy"]} | F1_DEM = {best_row["F1_DEM"]} | F1_GOP = {best_row["F1_GOP"]}')
print()
print('Matriz de confusión (filas = real, columnas = predicho):')
print(pd.DataFrame(cm, index=['real_GOP', 'real_DEM'], columns=['pred_GOP', 'pred_DEM']))
print()
print('Reporte completo:')
print(classification_report(y_test, y_pred_best, digits=4, zero_division=0))

Mejor modelo según F1-DEM: C=100.0
Accuracy = 0.9826 | F1_DEM = 0.9484 | F1_GOP = 0.9895

Matriz de confusión (filas = real, columnas = predicho):
          pred_GOP  pred_DEM
real_GOP       519         0
real_DEM        11       101

Reporte completo:


              precision    recall  f1-score   support

         DEM     1.0000    0.9018    0.9484       112
         GOP     0.9792    1.0000    0.9895       519

    accuracy                         0.9826       631
   macro avg     0.9896    0.9509    0.9689       631
weighted avg     0.9829    0.9826    0.9822       631



**Interpretación esperada:** con las 3 features crudas (`votes_gop`, `votes_dem`, `total_votes`) y un kernel lineal, el problema es **casi linealmente separable**: el hiperplano `votes_gop = votes_dem` ya da una accuracy muy alta. Las métricas serán elevadas para todos los valores de `C`; el efecto de `C` se notará sobre todo en el **número de vectores de soporte** y en la ubicación exacta del hiperplano, más que en la accuracy global.

---

## Fase 5 — Sistema de predicción para un condado nuevo

**Objetivo:** exponer una función reutilizable que reciba los votos crudos de un condado hipotético y devuelva `GOP` o `DEM`, aplicando internamente el `StandardScaler` entrenado.

### 5.1 Función `predict_county`

In [13]:
def predict_county(votes_gop: int, votes_dem: int, total_votes: int,
                      model, scaler) -> str:
    """Clasifica un condado hipotético a partir de sus votos crudos.

    Parameters
    ----------
    votes_gop, votes_dem, total_votes : int
        Votos crudos del condado (antes de escalar).
    model : sklearn.svm.SVC
        Modelo SVM lineal ya entrenado.
    scaler : sklearn.preprocessing.StandardScaler
        Escalador ya ajustado sobre el conjunto de entrenamiento.

    Returns
    -------
    str
        'GOP' o 'DEM'.
    """
    x = scaler.transform([[votes_gop, votes_dem, total_votes]])
    return model.predict(x)[0]

print('Función predict_county() definida.')

Función predict_county() definida.


### 5.2 Demostración con 3 condados inventados

In [14]:
examples = [
    ('GOP claro',     30000, 10000, 45000),
    ('DEM claro',      5000, 25000, 32000),
    ('Reñido',        15000, 14800, 30500),
]

print(f'Usando el mejor modelo: {best_name}\n')
print(f'{"Caso":<12} {"votes_gop":>10} {"votes_dem":>10} {"total":>8}   {"Predicción":>10}')
print('-' * 60)
for label, vg, vd, vt in examples:
    pred = predict_county(vg, vd, vt, best_model, scaler)
    print(f'{label:<12} {vg:>10,} {vd:>10,} {vt:>8,}   {pred:>10}')

Usando el mejor modelo: C=100.0

Caso          votes_gop  votes_dem    total   Predicción
------------------------------------------------------------
GOP claro        30,000     10,000   45,000          GOP
DEM claro         5,000     25,000   32,000          DEM
Reñido           15,000     14,800   30,500          GOP


**Verificación cualitativa:** los casos "GOP claro" y "DEM claro" deberían predecirse correctamente. El caso "reñido" puede caer a uno u otro lado según el modelo y el escalado; el SVM lineal tiende a separar por la diferencia `votes_gop - votes_dem`, así que con una diferencia de +200 debería clasificar como `GOP`.

**Reutilización:** la función `predict_county` puede importarse y usarse en otro script siempre que se le pase el `model` y el `scaler` entrenados (ambos objetos de scikit-learn son serializables con `joblib.dump` si se quisiera persistir).